In [1]:
pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.5 MB/s eta 0:00:00


In [2]:
pip install tensorflow

In [3]:
pip install scikit-learn

In [4]:
pip install pandas

In [8]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score
import kerastuner as kt

In [13]:
df = pd.read_csv('huge_1M_titanic.csv')

In [14]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1310,1,1,"Name1310, Miss. Surname1310",female,NaN,0,0,SOTON/O2 3101272,76.760165,NaN,C
1,1311,0,3,"Name1311, Col. Surname1311",male,29.0,0,0,223596,10.193097,NaN,S
2,1312,0,3,"Name1312, Mr. Surname1312",male,20.0,0,0,54636,12.029416,C83,C
3,1313,0,3,"Name1313, Mr. Surname1313",male,27.0,0,0,PC 17760,13.429448,NaN,S
4,1314,0,3,"Name1314, Mr. Surname1314",male,32.0,0,0,364512,4.840769,E33,C


In [15]:
df.drop(columns=['PassengerId', 'Name', 'Age', 'Ticket', 'Cabin'], inplace=True)

In [16]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
Embarked,2240


In [17]:
df.Sex = df['Sex'].astype('category')
# df.Embarked = df['Embarked'].astype('category')
df.Survived = df['Survived'].astype('int32')
df.Pclass = df['Pclass'].astype('int32')
df.SibSp = df['SibSp'].astype('int32')
df.Parch = df['Parch'].astype('int32')

In [18]:
df.Embarked = df.Embarked.replace({'S': 'Southampton', 'C': 'Chebourg', 'Q': 'Queenstown'})

In [19]:
df.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,Chebourg
1,0,3,male,0,0,10.193097,Southampton
2,0,3,male,0,0,12.029416,Chebourg
3,0,3,male,0,0,13.429448,Southampton
4,0,3,male,0,0,4.840769,Chebourg


In [20]:
df.dropna(subset=['Embarked'], inplace=True)

In [21]:
df['Fare'] = df['Fare'].astype('int8')

In [22]:
le = LabelEncoder()

df['Sex'] = le.fit_transform(df['Sex'])

In [23]:
ohe = OneHotEncoder(sparse_output=False)

Embarked = ohe.fit_transform(df[['Embarked']])

In [24]:
Embarked = pd.DataFrame(Embarked, columns=ohe.get_feature_names_out())

In [25]:
df = pd.concat([df.drop(columns=['Embarked']), Embarked], axis=1)

In [26]:
df.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Chebourg,Embarked_Queenstown,Embarked_Southampton
0,1.0,1.0,0.0,0.0,0.0,76.0,1.0,0.0,0.0
1,0.0,3.0,1.0,0.0,0.0,10.0,0.0,0.0,1.0
2,0.0,3.0,1.0,0.0,0.0,12.0,1.0,0.0,0.0
3,0.0,3.0,1.0,0.0,0.0,13.0,0.0,0.0,1.0
4,0.0,3.0,1.0,0.0,0.0,4.0,1.0,0.0,0.0


In [27]:
df.isna().sum()

,0
Survived,2234
Pclass,2234
Sex,2234
SibSp,2234
Parch,2234
Fare,2234
Embarked_Chebourg,2234
Embarked_Queenstown,2234
Embarked_Southampton,2234


In [34]:
df.isna().sum()

,0
Survived,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0
Embarked_Chebourg,0
Embarked_Queenstown,0
Embarked_Southampton,0


In [28]:
df.dropna(inplace=True)

In [31]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']

scale = StandardScaler()

df[num_cols] = scale.fit_transform(df[num_cols])

In [35]:
df.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Chebourg,Embarked_Queenstown,Embarked_Southampton
0,1.0,-1.568955,0.0,-0.462632,-0.469293,1.675910,1.0,0.0,0.0
1,0.0,0.824071,1.0,-0.462632,-0.469293,-0.234040,0.0,0.0,1.0
2,0.0,0.824071,1.0,-0.462632,-0.469293,-0.176163,1.0,0.0,0.0
3,0.0,0.824071,1.0,-0.462632,-0.469293,-0.147224,0.0,0.0,1.0
4,0.0,0.824071,1.0,-0.462632,-0.469293,-0.407672,1.0,0.0,0.0


In [36]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
model = Sequential(
    [
        Dense(128, input_shape=(X_train.shape[1],), activation='relu'), # First Hidden Layer
        Dense(64, activation='relu'), # First Hidden Layer
        Dense(32, activation='relu'), # Second Hidden Layer
        Dense(1, activation='sigmoid'), # Output Layer
    ]
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,521 (45.00 KB)

 Trainable params: 11,521 (45.00 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
import tensorflow

opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)

model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

In [45]:
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)

Epoch 1/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 69s 3ms/step - accuracy: 0.8487 - loss: 0.3368 - val_accuracy: 0.8481 - val_loss: 0.3378
Epoch 2/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 68s 3ms/step - accuracy: 0.8490 - loss: 0.3357 - val_accuracy: 0.8488 - val_loss: 0.3349
Epoch 3/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.8491 - loss: 0.3354 - val_accuracy: 0.8487 - val_loss: 0.3356
Epoch 4/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 84s 3ms/step - accuracy: 0.8490 - loss: 0.3356 - val_accuracy: 0.8479 - val_loss: 0.3340
Epoch 5/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 86s 3ms/step - accuracy: 0.8494 - loss: 0.3348 - val_accuracy: 0.8497 - val_loss: 0.3360
Epoch 6/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 76s 3ms/step - accuracy: 0.8493 - loss: 0.3350 - val_accuracy: 0.8492 - val_loss: 0.3370
Epoch 7/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.8492 - loss: 0.3354 - val_accuracy: 0.8482 - val_loss: 0.3373
Epoch 8/10
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 66s 3ms/step - accuracy: 

## Optimal Optimizer

SGD
RMSprop
Adam
AdamW
Adadelta
Adagrad
Adamax

In [50]:
def build_model(hp):
  model = Sequential(
      [
          Dense(64, input_shape=(X_train.shape[1],), activation='relu'),
          Dense(32, activation='relu'),
          Dense(1, activation='sigmoid')
      ]
  )
  optimizer = hp.Choice('optimizer', values=['sgd', 'rmsprop', 'adam', 'adamw', 'adadelta', 'adagrad', 'adamax'])
  model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
  return model

In [52]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)

In [54]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

Trial 5 Complete [00h 06m 00s]
val_accuracy: 0.8467650413513184

Best val_accuracy So Far: 0.8541781902313232
Total elapsed time: 00h 29m 16s


In [55]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [60]:
tuner.get_best_models()[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


<Sequential name=sequential, built=True>

In [56]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [57]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 67s 3ms/step - accuracy: 0.8550 - loss: 0.3271 - val_accuracy: 0.8520 - val_loss: 0.3329
Epoch 2/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.8551 - loss: 0.3265 - val_accuracy: 0.8547 - val_loss: 0.3284


## Choosing the most ideal number of nodes in Hidden Layer

In [58]:
def build_model(hp):
  nodes = hp.Int('nodes', 8, 128, step=8)
  activation = hp.Choice('activation', values=['tanh', 'sigmoid'])
  optimizer = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
  model = Sequential()
  model.add(Dense(nodes, input_shape=(X_train.shape[1],), activation='relu'))
  model.add(Dense(nodes, activation='relu'))
  model.add(Dense(1, activation=activation))
  model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
  return model

In [61]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='my_dir', project_name='intro_to_kt')

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [62]:
tuner.search(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Trial 3 Complete [00h 02m 37s]
val_accuracy: 0.8432493209838867

Best val_accuracy So Far: 0.850270688533783
Total elapsed time: 00h 07m 10s


In [64]:
tuner.get_best_hyperparameters()[0].values

{'nodes': 104, 'activation': 'sigmoid'}

In [65]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [66]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 71s 3ms/step - accuracy: 0.8514 - loss: 0.3351 - val_accuracy: 0.8503 - val_loss: 0.3383
Epoch 2/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 67s 3ms/step - accuracy: 0.8519 - loss: 0.3347 - val_accuracy: 0.8513 - val_loss: 0.3354


## Number of optimal number of hidden layer

In [72]:
from tensorflow.keras.layers import Input

def build_model(hp):

  model = Sequential()

  model.add(Input(shape=(X_train.shape[1],)))

  for i in range(hp.Int('num_layers', min_value=1, max_value=10, step=1)):
    model.add(Dense(64, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

In [73]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=2, directory='hidden', project_name='intro_to_hidden')

Reloading Tuner from hidden/intro_to_hidden/tuner0.json


In [74]:
tuner.search(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Trial 2 Complete [00h 03m 56s]
val_accuracy: 0.8531134128570557

Best val_accuracy So Far: 0.8531134128570557
Total elapsed time: 00h 09m 15s


In [75]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 9}

In [76]:
model = tuner.get_best_models(num_models=1)[0]


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [77]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 126s 5ms/step - accuracy: 0.8547 - loss: 0.3277 - val_accuracy: 0.8526 - val_loss: 0.3293
Epoch 2/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 118s 5ms/step - accuracy: 0.8550 - loss: 0.3267 - val_accuracy: 0.8535 - val_loss: 0.3281


## HyperParameter Tuning --->
1. Number of Hidden Layers
2. No. of Nodes in Hidden Layer
3. Optimal Optimizer
4. Dropout Layer Value

In [79]:
from tensorflow.keras.layers import Dropout

In [82]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(X_train.shape[1],)))
  hidden_layer = hp.Int('Hidden Layers', min_value=1, max_value=10, step=1)

  for i in range(hidden_layer):
    nodes = hp.Int('Nodes', min_value=8, max_value=128, step=8)
    model.add(Dense(nodes, activation='relu'))
    dropout = hp.Float('dropout', min_value=0.1, max_value=0.9, step=0.1)
    model.add(Dropout(dropout))

  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['sgd', 'rmsprop', 'adam', 'adamw', 'adadelta', 'adagrad', 'adamax'])
  model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

  return model


In [84]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='my_new', project_name='hyper')

In [85]:
tuner.search(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Trial 3 Complete [00h 02m 32s]
val_accuracy: 0.8113216161727905

Best val_accuracy So Far: 0.8512752056121826
Total elapsed time: 00h 09m 19s


In [86]:
tuner.get_best_hyperparameters()[0].values

{'Hidden Layers': 3, 'Nodes': 104, 'dropout': 0.1, 'optimizer': 'rmsprop'}

In [87]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [88]:
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 82s 3ms/step - accuracy: 0.8477 - loss: 0.3526 - val_accuracy: 0.8501 - val_loss: 0.3413
Epoch 2/2
24889/24889 ━━━━━━━━━━━━━━━━━━━━ 81s 3ms/step - accuracy: 0.8488 - loss: 0.3523 - val_accuracy: 0.8511 - val_loss: 0.3380
